# Milestone 4 — Continuous batching

Schedule every decode step: evict finished requests, admit waiting ones into free slots.
Compare the same arrival workload under **static waves** vs **continuous** scheduling.

In [ ]:
!git clone https://github.com/Jayaprakash-030/tiny-inference-engine.git
%cd tiny-inference-engine
!pip install -q -e .

In [ ]:
from engine import load
from engine.continuous import (
    check_decode_batch_matches_run_one,
    check_run_one_matches_cached,
    measure_pair,
    print_pair,
    sweep_continuous,
)
from engine.results import save

rt = load()
rt.describe()

### Correctness

In [ ]:
assert check_run_one_matches_cached(rt)
assert check_decode_batch_matches_run_one(rt)

### One pair (smoke) then sweep + save

In [ ]:
s, c = measure_pair(rt, n_requests=16, max_slots=4, max_new_tokens=64)
print_pair(s, c)

In [ ]:
m4 = sweep_continuous(
    rt,
    slot_sizes=(2, 4, 8, 16),
    n_requests=16,
    max_new_tokens=64,
)
save(m4)

### Plot static vs continuous

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

Path("benchmarks/plots").mkdir(parents=True, exist_ok=True)

static = [r for r in m4 if r["mode"] == "static"]
cont = [r for r in m4 if r["mode"] == "continuous"]
slots = [r["max_slots"] for r in static]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title in [
    (axes[0], "tokens_per_sec", "Throughput (tok/s)"),
    (axes[1], "mean_latency_ms", "Mean latency (ms)"),
    (axes[2], "slot_util_pct", "Slot utilization (%)"),
]:
    ax.plot(slots, [r[key] for r in static], marker="o", label="static")
    ax.plot(slots, [r[key] for r in cont], marker="o", label="continuous")
    ax.set_title(title)
    ax.set_xlabel("max_slots")
    ax.set_xscale("log", base=2)
    ax.set_xticks(slots)
    ax.set_xticklabels(slots)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("benchmarks/plots/m4_continuous.png", dpi=140)
plt.show()